In [ ]:
import sqlite3
import pandas as pd

# le em csv e tranforma em uma tabela
df = pd.read_csv("../data/train.csv")

# 2. Abre (ou cria) a conexão com o banco SQLite (.. = volta uma pasta(scripts))
conexao = sqlite3.connect("../credito.db")

# 3. Escreve o DataFrame como uma tabela dentro do banco
df.to_sql("clientes", conexao, if_exists="replace", index=False)

conexao.close()

In [ ]:
conexao = sqlite3.connect("../credito.db") # conecta na tabela
cursor = conexao.cursor() # criaa o canal para conexões com o banco
cursor.execute("SELECT COUNT(*) FROM clientes") 
print(cursor.fetchone())
conexao.close()

(1, 0, 0.957151019, 40, 0, 0.1218762009999999, 2600.0, 4, 0, 0, 0, 1.0)


In [ ]:
# Pergunta 1: Distribuição da variável alvo
#Quantos clientes são inadimplentes vs não inadimplentes?

conexao = sqlite3.connect("../credito.db")
cursor = conexao.cursor()
cursor.execute("SELECT SeriousDlqin2yrs, COUNT(*) FROM CLIENTES GROUP BY SeriousDlqin2yrs")
resultado = cursor.fetchall()
print(resultado)

# 0 = não inadimplentes

[(0, 97855), (1, 6950)]


In [ ]:
# Pergunta 2: Valores nulos em MonthlyIncome e NumberOfDependents
#Quantos clientes têm renda ou número de dependentes não informados?

cursor.execute("""
    SELECT 
        SUM(CASE WHEN MonthlyIncome IS NULL THEN 1 ELSE 0 END) AS nulos_renda,
        SUM(CASE WHEN NumberOfDependents IS NULL THEN 1 ELSE 0 END) AS nulos_dependentes
    FROM clientes
""")
resultado = cursor.fetchone()
print(resultado)

# 0 é verdadeiro



(20781, 2749)


MonthlyIncome (renda mensal): 20.781 valores nulos — isso é bastante, quase 20% da base (20.781 ÷ 104.805 ≈ 19,8%). 
Vai exigir uma decisão de tratamento lá na frente (preencher com mediana, remover, ou criar uma categoria "não informado".


NumberOfDependents (nº de dependentes): 2.749 valores nulos — bem menos grave, cerca de 2,6% da base. 
Mais fácil de tratar (dá pra assumir 0 dependentes, por exemplo, ou usar a moda).

In [ ]:

# Pergunta 2.1: Outliers de idade
#Qual a idade mínima, máxima e média da base? Existem valores absurdos?

cursor.execute("""
    SELECT MIN(age), MAX(age), AVG(age)
    FROM clientes
""")
resultado = cursor.fetchone()
print(resultado)

(0, 109, 52.35112828586423)


In [ ]:
cursor.execute("""
    SELECT COUNT(*)
    FROM clientes
    WHERE age = 0
""")
resultado = cursor.fetchone()
print(resultado)

(0,)


In [ ]:
# Valores fora do esperado
#RevolvingUtilizationOfUnsecuredLines e DebtRatio deveriam ser proporções (0 a 1). Existem valores muito acima disso?


cursor.execute("""
    SELECT 
        MIN(RevolvingUtilizationOfUnsecuredLines), 
        MAX(RevolvingUtilizationOfUnsecuredLines),
        MIN(DebtRatio),
        MAX(DebtRatio)
    FROM clientes
""")
resultado = cursor.fetchone()
print(resultado)

(0.0, 29110.0, 0.0, 329664.0)


1- Quanto do limite disponível (cartão de crédito, cheque especial) o cliente já está usando. 
É uma proporção — em teoria entre 0 e 1 (0% a 100%), mas às vezes aparece valores estranhos acima disso, o que vale investigar.

RevolvingUtilizationOfUnsecuredLines = saldo usado ÷ limite total disponível

(Matematicamente, isso é a definição de "taxa de utilização" — e por construção, 
esse tipo de cálculo dá um número entre 0 e 1 (0% a 100% do limite usado). 
Só ultrapassaria um pouco o 1 num caso raro de alguém estourar o limite (ex: usar 105% por causa de juros, o que ainda seria perto de 1, tipo 1.05)

2- Proporção entre despesas mensais (dívidas, pensão, etc) e renda mensal. Quanto maior, mais endividado o cliente está em relação ao que ganha.

DebtRatio = despesas mensais ÷ renda mensal

(Análise realizada para identificação de erros de qualidade de dados, o problema é que deveria ser uma proporçaõ de 0 a 1)

In [21]:
cursor.execute("""
    SELECT COUNT(*)
    FROM clientes
    WHERE Debtratio > 2
""")
resultado = cursor.fetchone()
print(resultado)

(21688,)


In [ ]:
# renda média dos inadimplentes vs não inadimplentes.

cursor.execute("""
    SELECT SeriousDlqin2yrs, AVG(MonthlyIncome), COUNT(*)
    FROM clientes
    WHERE MonthlyIncome IS NOT NULL
    GROUP BY SeriousDlqin2yrs
""")
resultado = cursor.fetchall()
print(resultado)

# Não inadimplentes (0): renda média de R$ 6.763,40 (78.242 clientes com renda informada)
# Inadimplentes (1): renda média de R$ 5.616,21 (5.782 clientes com renda informada)

[(0, 6763.395145829606, 78242), (1, 5616.206157039087, 5782)]


In [23]:
# A taxa de inadimplência muda por faixa etária?

cursor.execute("""
    SELECT 
        CASE 
            WHEN age < 30 THEN '18-29'
            WHEN age < 45 THEN '30-44'
            WHEN age < 60 THEN '45-59'
            ELSE '60+'
        END AS faixa_etaria,
        AVG(SeriousDlqin2yrs) AS taxa_inadimplencia,
        COUNT(*) AS total_clientes
    FROM clientes
    GROUP BY faixa_etaria
""")
resultado = cursor.fetchall()
print(resultado)

[('18-29', 0.114004914004914, 6105), ('30-44', 0.0942715048811936, 27145), ('45-59', 0.07083410885692254, 37609), ('60+', 0.030371766923938018, 33946)]


AVG(SeriousDlqin2yrs) — esse é um truque útil: como SeriousDlqin2yrs só tem valores 0 ou 1, 
calcular a média dela é matematicamente igual a calcular a proporção de inadimplentes naquele grupo. Ex: se a média der 0.10, significa que 10% daquele grupo é inadimplente.

Padrão bem consistente: quanto mais velho o cliente, menor a taxa de inadimplência. Os mais jovens (18-29) têm quase 4x mais chance de inadimplir que os mais velhos (60+). Isso confirma o que a literatura de crédito já mostra — pessoas mais jovens costumam ter menos histórico de crédito, renda mais instável e menos "colchão" financeiro, enquanto clientes mais velhos costumam ter situação financeira mais estável e comportamento de pagamento mais consolidado.

In [24]:
# Quem já teve atraso de 90+ dias (NumberOfTimes90DaysLate > 0) tem taxa de inadimplência bem mais alta agora?

cursor.execute("""
    SELECT 
        CASE 
            WHEN NumberOfTimes90DaysLate = 0 THEN 'Nunca atrasou 90+ dias'
            ELSE 'Já atrasou 90+ dias'
        END AS grupo,
        AVG(SeriousDlqin2yrs) AS taxa_inadimplencia,
        COUNT(*) AS total_clientes
    FROM clientes
    GROUP BY grupo
""")
resultado = cursor.fetchall()
print(resultado)

[('Já atrasou 90+ dias', 0.4155003498950315, 5716), ('Nunca atrasou 90+ dias', 0.046170614296238734, 99089)]


Quem já teve um atraso grave de 90+ dias tem uma taxa de inadimplência quase 9x maior que quem nunca teve.

Só 5.716 clientes (5,4% da base) já tiveram esse tipo de atraso — é um grupo pequeno, mas com comportamento muito diferente do resto.



In [25]:
# Mais linhas de crédito abertas está associado a mais ou menos inadimplência?

cursor.execute("""
    SELECT 
        CASE 
            WHEN NumberOfOpenCreditLinesAndLoans = 0 THEN '0 linhas'
            WHEN NumberOfOpenCreditLinesAndLoans <= 5 THEN '1-5 linhas'
            WHEN NumberOfOpenCreditLinesAndLoans <= 10 THEN '6-10 linhas'
            ELSE '11+ linhas'
        END AS faixa_linhas,
        AVG(SeriousDlqin2yrs) AS taxa_inadimplencia,
        COUNT(*) AS total_clientes
    FROM clientes
    GROUP BY faixa_linhas
""")
resultado = cursor.fetchall()
print(resultado)

[('0 linhas', 0.2471042471042471, 1295), ('1-5 linhas', 0.07589099875116077, 31229), ('11+ linhas', 0.0645524737382775, 30177), ('6-10 linhas', 0.05491164734942048, 42104)]


O padrão é uma espécie de "U": quem não tem nenhuma linha de crédito aberta tem risco disparadamente maior (24,7%) — provavelmente por falta de histórico de crédito consolidado, o que é um sinal de risco em si (bancos costumam desconfiar de quem não tem "nome no mercado"). Depois o risco cai bastante na faixa 1-5 e atinge o mínimo em 6-10 linhas (5,5%) — um "ponto ideal" de diversificação de crédito. Na faixa 11+ o risco sobe de novo, levemente, talvez por sobrecarga de compromissos financeiros.

Vale notar: o grupo "0 linhas" é pequeno (só 1.295 clientes, 1,2% da base) — a taxa é confiável, mas é um grupo minoritário, então não domina o resultado geral.

In [26]:
#Clientes com imóveis financiados tendem a ser menos inadimplentes?

cursor.execute("""
    SELECT 
        CASE 
            WHEN NumberRealEstateLoansOrLines = 0 THEN 'Sem imóvel financiado'
            ELSE 'Com imóvel financiado'
        END AS grupo,
        AVG(SeriousDlqin2yrs) AS taxa_inadimplencia,
        COUNT(*) AS total_clientes
    FROM clientes
    GROUP BY grupo
""")
resultado = cursor.fetchall()
print(resultado)

[('Com imóvel financiado', 0.056664785466969694, 65561), ('Sem imóvel financiado', 0.0824329833859953, 39244)]


Quem tem pelo menos um financiamento imobiliário tem risco ~30% menor que quem não tem. Faz sentido: 
ter um financiamento imobiliário aprovado já é um filtro em si (o banco avaliou e aprovou o crédito), 
e manter esse compromisso em dia por anos é sinal de estabilidade financeira consistente.

In [ ]:
# Recapitulando os achados até aqui, do mais forte pro mais fraco:

# Atraso de 90+ dias no passado → sinal mais forte (4,6% vs 41,6%)
# Nenhuma linha de crédito aberta → risco bem alto isolado (24,7%)
# Idade → cai de forma consistente conforme envelhece (11,4% → 3,0%)
# Imóvel financiado → reduz o risco (~30% menor)
 #Renda mensal → diferença real mas mais moderada (~17% menor pra inadimplentes)

In [ ]:
# Média de Renda

cursor.execute("""
    SELECT AVG(MonthlyIncome)
    FROM clientes
    WHERE MonthlyIncome IS NOT NULL
""")
resultado = cursor.fetchone()
print(resultado)

(6684.452858707036,)


In [ ]:
# Mediana (Truque)

cursor.execute("""
    SELECT MonthlyIncome
    FROM clientes
    WHERE MonthlyIncome IS NOT NULL
    ORDER BY MonthlyIncome
    LIMIT 1
    OFFSET (
        SELECT COUNT(*) / 2
        FROM clientes
        WHERE MonthlyIncome IS NOT NULL
    )
""")
resultado = cursor.fetchone()
print(resultado)

#OFFSET diz "pule as N primeiras linhas antes de começar a trazer resultado". 
# Combinado com LIMIT 1, a lógica vira: "pule até a metade dos dados, e me traga a próxima linha" — ou seja, exatamente o valor do meio.

#Média: R$ 6.684,45
#Mediana: R$ 5.400,00

(5400.0,)


In [29]:
# Quantos clientes têm 0 dependentes vs 1+.

cursor.execute("""
    SELECT 
        CASE 
            WHEN NumberOfDependents = 0 OR NumberOfDependents IS NULL THEN '0 dependentes'
            ELSE '1+ dependentes'
        END AS grupo,
        COUNT(*) AS total_clientes
    FROM clientes
    GROUP BY grupo
""")
resultado = cursor.fetchall()
print(resultado)

[('0 dependentes', 63458), ('1+ dependentes', 41347)]
